
# Hybrid Quantum Physics-Informed Neural Network (QPINN)
## 2D Crystal Growth with Phase-Field Physics (10–16 Qubits)

This revised notebook corrects the main conceptual issues in the original draft and expands it into a more defensible **hybrid quantum analogue of SI-LB-PINN** for thermal–fluid / phase-field crystal growth.

We model

$$
\mathbf{y}(x,y)=\big(u(x,y),v(x,y),p(x,y),c(x,y),\phi(x,y)\big),
$$

where

- $u,v$ are the velocity components,
- $p$ is the pressure,
- $c$ is concentration or temperature-like transported scalar,
- $\phi$ is the phase field.

Interpretation of the phase field:

- $\phi \approx 1$ : solid,
- $\phi \approx -1$ : liquid,
- $\phi \approx 0$ : diffuse interface.

The classical reference idea is the **SI-LB-PINN** paper for Czochralski silicon single-crystal growth, which combines **enhanced spatial information** with **adaptive loss balancing** to improve stability and accuracy on coupled thermal-fluid fields. The paper reports better accuracy than baseline enhanced PINN variants on a square-cavity validation problem and then applies the method to 2D axisymmetric CZ melt thermal-fluid coupling. citeturn778797view0turn847676search0



## 1. What was wrong or incomplete in the original notebook

The original notebook had a good high-level direction, but several issues should be fixed before calling it a hybrid QPINN:

### Main corrections

1. **The anisotropic chemical potential was oversimplified.**
   If $\epsilon$ depends on interface angle $\theta$, the term should be written in divergence form,
   $$
   \mu = -\nabla\cdot\big(\epsilon(\theta)^2\nabla\phi\big) + \phi(\phi^2-1) - 2\lambda_c c\phi,
   $$
   not just $-\epsilon(\theta)^2\nabla^2\phi$ unless $\epsilon$ is treated as constant.

2. **The loss only constrained the phase field.**
   Since the outputs are $(u,v,p,c,\phi)$, the PINN must include residuals for incompressibility, momentum, transport, and phase-field evolution.

3. **The quantum layer detached the inputs and weights.**
   In the original `LocalQuantumLayer`, `detach().cpu().numpy()` breaks gradient flow, so the model is not trainable end-to-end.

4. **The quantum estimator was called one observable at a time.**
   That is computationally inefficient. Local observables should be batched in one estimator call.

5. **The latent quantum inputs were unbounded.**
   Rotation angles should be scaled, e.g. to $[-\pi,\pi]$.

6. **The formulation was not yet SI-LB-like.**
   The SI idea means reinjecting spatial coordinates into deeper hidden representations; the LB idea means adaptive weighting across residual groups.

7. **The notebook used a statevector primitive while discussing few-shot execution.**
   Statevector simulation is noiseless and shot-free, so it does not validate the hardware claim. For hardware, a shot-based estimator/sampler and explicit mitigation are needed.



## 2. A more consistent PDE system for hybrid QPINN

A simplified steady coupled system can be written as

### Incompressibility
$$
R_{\mathrm{div}} = u_x + v_y = 0
$$

### Momentum
$$
R_u = u u_x + v u_y + p_x - \nu \nabla^2 u - f_u(c,\phi),
$$
$$
R_v = u v_x + v v_y + p_y - \nu \nabla^2 v - f_v(c,\phi).
$$

### Scalar transport
$$
R_c = u c_x + v c_y - D_c \nabla^2 c - S_c(\phi,c).
$$

### Phase-field evolution
For a stationary interface fit one can use a relaxed Allen–Cahn form
$$
R_\phi = u\phi_x + v\phi_y + M_\phi\mu,
$$
with
$$
\mu = -\nabla\cdot\big(\epsilon(\theta)^2\nabla\phi\big) + \phi(\phi^2-1) - 2\lambda_c c\phi,
$$
$$
\theta = \operatorname{atan2}(\phi_y,\phi_x), \qquad
\epsilon(\theta)=\epsilon_0\big(1+\delta\cos(m\theta)\big).
$$

A Stefan-like interfacial consistency penalty can be added as
$$
R_S = \mu - \lambda_T c\lVert \nabla \phi \rVert.
$$

Then the PDE loss is
$$
\mathcal{L}_{\mathrm{PDE}} = \lambda_{\mathrm{div}}\Vert R_{\mathrm{div}}\Vert_2^2
+ \lambda_u\Vert R_u\Vert_2^2
+ \lambda_v\Vert R_v\Vert_2^2
+ \lambda_c\Vert R_c\Vert_2^2
+ \lambda_\phi\Vert R_\phi\Vert_2^2
+ \lambda_S\Vert R_S\Vert_2^2.
$$

Boundary, interface, and data terms are then added in the usual PINN way.



## 3. From SI-LB-PINN to hybrid quantum SI-LB-QPINN

The 2025 SI-LB-PINN idea has two ingredients: spatial-information enhancement and adaptive loss balancing. citeturn778797view0

A clean hybrid upgrade is:

### (a) Spatial-information enhancement
Use coordinate reinjection:
$$
z^{(0)} = [x,y], \qquad
h^{(\ell+1)} = \sigma\big(W^{(\ell)}[h^{(\ell)},x,y] + b^{(\ell)}\big).
$$
This preserves the SI idea from the classical paper.

### (b) Quantum latent feature map
Instead of a purely classical hidden block, define a quantum feature map
$$
q(x,y) = \big(\langle Z_1\rangle,\ldots,\langle Z_n\rangle\big) \in \mathbb{R}^{n_q},
$$
where the circuit is driven by a low-dimensional latent code
$$
a(x,y)=g_\eta(x,y)\in[-\pi,\pi]^d,
$$
and the overall model becomes
$$
\hat y(x,y)=f_\omega\Big(h_{\mathrm{SI}}(x,y),\; q\big(a(x,y);\vartheta\big)\Big).
$$

### (c) Adaptive loss balancing
Use trainable or dynamically updated weights,
$$
\mathcal L = \sum_{k=1}^{K} \lambda_k \mathcal L_k,
$$
with updates based on residual magnitude or gradient norm balancing, for example
$$
\lambda_k^{(t+1)} \propto \frac{1}{\operatorname{EMA}(\mathcal L_k^{(t)})+\varepsilon}
\quad\text{or}\quad
\lambda_k^{(t+1)} \propto \frac{\bar g^{(t)}}{g_k^{(t)}+\varepsilon},
$$
where $g_k$ is the gradient norm contributed by loss term $k$.

This keeps the **SI-LB** strengths while replacing part of the latent representation by a quantum feature layer.


In [ ]:

import math
import numpy as np
import torch
import torch.nn as nn

# Optional Qiskit imports. The notebook remains readable even if Qiskit is not installed.
try:
    from qiskit import QuantumCircuit
    from qiskit.circuit import Parameter
    from qiskit.quantum_info import SparsePauliOp
    from qiskit.primitives import StatevectorEstimator
    QISKIT_AVAILABLE = True
except Exception as e:
    QuantumCircuit = None
    Parameter = None
    SparsePauliOp = None
    StatevectorEstimator = None
    QISKIT_AVAILABLE = False
    print(f"Qiskit unavailable in this environment: {e}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


In [ ]:

# Configuration
N_QUBITS = 12           # valid target range: 10 to 16
N_Q_LAYERS = 3
LATENT_Q_DIM = 4        # number of re-uploaded data features
HIDDEN = 64

# Physics coefficients (illustrative placeholders)
NU = 1e-2
D_C = 1e-2
M_PHI = 1.0
EPS0 = 1e-2
DELTA_ANISO = 0.05
ANISO_M = 4
LAMBDA_C = 1.0
LAMBDA_T = 1.0

# Collocation sizes
N_BULK = 128
N_BC = 64
N_INTERFACE = 128



## 4. Hybrid architecture with SI-style coordinate reinjection

This block explicitly reinjects $(x,y)$ at each stage, matching the spirit of the SI-LB-PINN paper. citeturn778797view0


In [ ]:

class SIBlock(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim + 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
        )

    def forward(self, h, xy):
        return self.net(torch.cat([h, xy], dim=-1))


class SpatialEncoder(nn.Module):
    def __init__(self, hidden_dim=64, depth=3):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(2, hidden_dim), nn.Tanh())
        self.blocks = nn.ModuleList([SIBlock(hidden_dim, hidden_dim) for _ in range(depth)])

    def forward(self, xy):
        h = self.inp(xy)
        for block in self.blocks:
            h = h + block(h, xy)
        return h



## 5. Corrected quantum ansatz

For 10–16 qubits, a safer design is a **shallow local ansatz** with:

1. data re-uploading,
2. trainable single-qubit rotations,
3. alternating local entanglers.

A good template is
$$
U(a,\vartheta)=\prod_{\ell=1}^{L}
U_{\mathrm{ent}}^{(\ell)}
\Big(\prod_{j=1}^{n_q} R_Z(\alpha_{\ell,j})R_Y(a_{j\bmod d})R_X(\beta_{\ell,j})\Big).
$$

Two important design choices:

- use **local observables** rather than a global many-body observable,
- keep depth modest to reduce both noise and barren-plateau risk.


In [ ]:

def build_quantum_ansatz(n_qubits, n_layers, feature_dim=4):
    if not QISKIT_AVAILABLE:
        raise RuntimeError("Qiskit is required to build the circuit.")

    qc = QuantumCircuit(n_qubits)
    input_params = [Parameter(f"x{i}") for i in range(feature_dim)]
    weight_params = []

    for layer in range(n_layers):
        # data re-uploading + trainable local block
        for q in range(n_qubits):
            xq = input_params[q % feature_dim]
            qc.ry(xq, q)
            p1 = Parameter(f"thx_{layer}_{q}")
            p2 = Parameter(f"thz_{layer}_{q}")
            qc.rx(p1, q)
            qc.rz(p2, q)
            weight_params.extend([p1, p2])

        # alternating nearest-neighbor entanglers (brickwork)
        start = layer % 2
        for q in range(start, n_qubits - 1, 2):
            qc.cz(q, q + 1)

    return qc, input_params, weight_params


def build_local_observables(n_qubits):
    if not QISKIT_AVAILABLE:
        raise RuntimeError("Qiskit is required to build observables.")
    obs = []
    for i in range(n_qubits):
        pauli = ["I"] * n_qubits
        pauli[n_qubits - 1 - i] = "Z"
        obs.append(SparsePauliOp.from_list([("".join(pauli), 1.0)]))
    return obs



## 6. Important training note

A true trainable QPINN needs a differentiable quantum layer. The original notebook was not differentiable because it converted tensors to NumPy and back.

For actual end-to-end training, use one of these routes:

- **Qiskit Machine Learning** with `EstimatorQNN` + `TorchConnector`,
- **parameter-shift gradients** for quantum weights and PyTorch autograd for classical weights,
- **SPSA / 2-SPSA** for quantum parameters when shot noise is significant.

The code below is therefore a **correct forward-pass prototype** for analysis and architecture design, not a complete gradient-safe training layer.


In [ ]:

class QuantumFeatureExtractor(nn.Module):
    """
    Forward-pass prototype only.
    For real training, replace with EstimatorQNN/TorchConnector or a custom
    parameter-shift / SPSA wrapper.
    """
    def __init__(self, qc, input_params, weight_params, n_qubits):
        super().__init__()
        self.qc = qc
        self.input_params = input_params
        self.weight_params = weight_params
        self.n_qubits = n_qubits
        self.observables = build_local_observables(n_qubits) if QISKIT_AVAILABLE else None
        self.estimator = StatevectorEstimator() if QISKIT_AVAILABLE else None
        self.weights = nn.Parameter(0.02 * torch.randn(len(weight_params)))

    def _scale_inputs(self, x):
        return math.pi * torch.tanh(x)

    @torch.no_grad()
    def forward(self, x):
        if not QISKIT_AVAILABLE:
            raise RuntimeError("Qiskit is not available in this environment.")

        x = self._scale_inputs(x)
        outputs = []
        param_order = self.input_params + self.weight_params
        theta = self.weights.detach().cpu().numpy().tolist()

        for xi in x.detach().cpu().numpy():
            values = list(xi) + theta
            bind = {p: v for p, v in zip(param_order, values)}
            circ = self.qc.assign_parameters(bind)

            # Batch all local observables in one primitive call.
            pubs = [(circ, obs) for obs in self.observables]
            job = self.estimator.run(pubs)
            res = job.result()
            row = [float(r.data.evs) for r in res]
            outputs.append(row)

        return torch.tensor(outputs, dtype=torch.float32, device=x.device)


In [ ]:

class HybridSILBQPINN(nn.Module):
    def __init__(self, qlayer, n_qubits, hidden_dim=64):
        super().__init__()
        self.encoder = SpatialEncoder(hidden_dim=hidden_dim, depth=3)
        self.pre_q = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.Tanh(),
            nn.Linear(32, LATENT_Q_DIM),
        )
        self.q = qlayer
        self.post = nn.Sequential(
            nn.Linear(hidden_dim + n_qubits, 128),
            nn.Tanh(),
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 5),
        )

    def forward(self, xy):
        h = self.encoder(xy)
        q_in = self.pre_q(h)
        q_out = self.q(q_in)
        return self.post(torch.cat([h, q_out], dim=-1))



## 7. Physics terms with corrected anisotropy handling


In [ ]:

def anisotropic_epsilon(phi_x, phi_y):
    theta = torch.atan2(phi_y, phi_x + 1e-8)
    return EPS0 * (1.0 + DELTA_ANISO * torch.cos(ANISO_M * theta))


def anisotropic_flux(phi_x, phi_y):
    eps = anisotropic_epsilon(phi_x, phi_y)
    return eps**2 * phi_x, eps**2 * phi_y


def phase_field_mu(x, phi, phi_x, phi_y, c):
    # divergence form: -div(eps(theta)^2 grad phi)
    fx, fy = anisotropic_flux(phi_x, phi_y)

    grad_fx = torch.autograd.grad(
        fx, x, torch.ones_like(fx), create_graph=True, retain_graph=True
    )[0]
    grad_fy = torch.autograd.grad(
        fy, x, torch.ones_like(fy), create_graph=True, retain_graph=True
    )[0]

    div_term = grad_fx[:, 0] + grad_fy[:, 1]
    return -div_term + phi * (phi**2 - 1.0) - 2.0 * LAMBDA_C * c * phi


def stefan_residual(mu, phi_x, phi_y, c):
    grad_mag = torch.sqrt(phi_x**2 + phi_y**2 + 1e-8)
    return mu - LAMBDA_T * c * grad_mag


In [ ]:

def grad_scalar(y, x):
    return torch.autograd.grad(
        y, x, torch.ones_like(y), create_graph=True, retain_graph=True
    )[0]


def laplacian(y, x):
    g = grad_scalar(y, x)
    gx = torch.autograd.grad(g[:, 0], x, torch.ones_like(g[:, 0]), create_graph=True, retain_graph=True)[0][:, 0]
    gy = torch.autograd.grad(g[:, 1], x, torch.ones_like(g[:, 1]), create_graph=True, retain_graph=True)[0][:, 1]
    return gx + gy


In [ ]:

def residuals(model, x):
    out = model(x)
    u, v, p, c, phi = out.T

    gu = grad_scalar(u, x)
    gv = grad_scalar(v, x)
    gp = grad_scalar(p, x)
    gc = grad_scalar(c, x)
    gphi = grad_scalar(phi, x)

    u_x, u_y = gu[:, 0], gu[:, 1]
    v_x, v_y = gv[:, 0], gv[:, 1]
    p_x, p_y = gp[:, 0], gp[:, 1]
    c_x, c_y = gc[:, 0], gc[:, 1]
    phi_x, phi_y = gphi[:, 0], gphi[:, 1]

    lap_u = laplacian(u, x)
    lap_v = laplacian(v, x)
    lap_c = laplacian(c, x)

    # simplified placeholders for body-force/source coupling
    f_u = torch.zeros_like(u)
    f_v = torch.zeros_like(v)
    s_c = torch.zeros_like(c)

    mu = phase_field_mu(x, phi, phi_x, phi_y, c)

    r_div = u_x + v_y
    r_u = u * u_x + v * u_y + p_x - NU * lap_u - f_u
    r_v = u * v_x + v * v_y + p_y - NU * lap_v - f_v
    r_c = u * c_x + v * c_y - D_C * lap_c - s_c
    r_phi = u * phi_x + v * phi_y + M_PHI * mu
    r_stefan = stefan_residual(mu, phi_x, phi_y, c)

    return {
        "div": r_div,
        "mom_u": r_u,
        "mom_v": r_v,
        "scalar": r_c,
        "phase": r_phi,
        "stefan": r_stefan,
    }



## 8. Adaptive loss balancing in SI-LB spirit

A simple practical version is to maintain exponentially smoothed term magnitudes and reweight inversely:
$$
\lambda_k \leftarrow \frac{1}{\text{EMA}(\mathcal L_k)+\varepsilon},
\qquad
\lambda_k \leftarrow \frac{\lambda_k}{\sum_j \lambda_j}K.
$$

This is not the only option, but it is a clean way to reflect the **LB** idea from SI-LB-PINN. citeturn778797view0


In [ ]:

class AdaptiveLossBalancer:
    def __init__(self, keys, beta=0.95, eps=1e-8):
        self.keys = list(keys)
        self.beta = beta
        self.eps = eps
        self.ema = {k: 1.0 for k in self.keys}

    def weights(self, losses):
        for k in self.keys:
            val = float(losses[k].detach().cpu())
            self.ema[k] = self.beta * self.ema[k] + (1 - self.beta) * val
        raw = {k: 1.0 / (self.ema[k] + self.eps) for k in self.keys}
        s = sum(raw.values())
        K = len(raw)
        return {k: K * raw[k] / s for k in raw}


def total_loss(model, x, balancer=None):
    res = residuals(model, x)
    losses = {k: (v**2).mean() for k, v in res.items()}

    if balancer is None:
        w = {k: 1.0 for k in losses}
    else:
        w = balancer.weights(losses)

    total = sum(w[k] * losses[k] for k in losses)
    return total, losses, w



## 9. Why local observables justify larger circuits with fewer shots

This is the most important mathematical point for a hybrid quantum upgrade.

Suppose each quantum feature is a local Pauli expectation
$$
q_j = \langle Z_j \rangle, \qquad j=1,\dots,n_q.
$$
With $S$ shots, the sample estimator $\hat q_j$ satisfies
$$
\operatorname{Var}(\hat q_j)=\frac{1-q_j^2}{S}\le \frac{1}{S}.
$$
Therefore the mean-squared feature error scales as
$$
\mathbb E\|\hat q-q\|_2^2 = \sum_{j=1}^{n_q}\operatorname{Var}(\hat q_j)
\le \frac{n_q}{S}.
$$
If the classical post-network is $L_f$-Lipschitz, then the output perturbation obeys
$$
\mathbb E\|f(h,\hat q)-f(h,q)\|_2^2 \le L_f^2\frac{n_q}{S}.
$$

### Consequence
For **local observables**, the shot noise grows only linearly with the number of measured qubits and decays as $1/S$. It does **not** directly depend on the full Hilbert-space dimension $2^{n_q}$.

That is why a 10–16 qubit circuit can still be reasonable if

- the ansatz is shallow,
- the cost is local,
- the measured feature vector is local,
- the post-network compresses noisy quantum features robustly.

### Why this is better than global costs
If one used a global observable or highly nonlocal fidelity-like cost, effective signal can decay rapidly with system size and depth, while gradient concentration can worsen. By keeping the cost local, one stays closer to the regime where barren-plateau scaling is much milder.

### Why fewer shots can still work
For PINNs, the total optimization noise has two parts:
$$
\operatorname{Var}(\widehat{\nabla \mathcal L})
\approx \operatorname{Var}_{\text{collocation}} + \operatorname{Var}_{\text{measurement}}.
$$
If collocation stochasticity already dominates early in training, increasing quantum shots beyond the level where
$$
\operatorname{Var}_{\text{measurement}} \lesssim \operatorname{Var}_{\text{collocation}}
$$
brings diminishing returns. This gives a principled reason to use **fewer shots early** and reallocate shots later only to the most sensitive residual terms.



## 10. Shot allocation rule for PINN residuals

Suppose the loss decomposes as
$$
\mathcal L = \sum_{k=1}^{K}\lambda_k\mathcal L_k,
$$
and residual group $k$ has measurement variance approximately $\sigma_k^2/S_k$ under $S_k$ shots.

Under a fixed shot budget
$$
\sum_{k=1}^{K} S_k = S_{\text{tot}},
$$
minimizing
$$
\sum_{k=1}^{K}\lambda_k^2\frac{\sigma_k^2}{S_k}
$$
by Lagrange multipliers gives
$$
S_k^\star \propto \lambda_k \sigma_k.
$$

So the mathematically justified policy is:

- allocate **more shots** to high-weight, high-variance residual groups,
- allocate **fewer shots** to stable or weakly weighted terms,
- update this allocation during training.

This is one of the cleanest ways to argue for **larger circuits with fewer total shots** than a naive uniform-shot strategy.



## 11. Error mitigation recommendations for 10–16 qubits

For hardware-oriented execution, the following mitigation stack is the most defensible:

### (a) Measurement calibration / readout mitigation
Use assignment-matrix calibration or M3-type mitigation to reduce SPAM bias on local $Z_j$ expectations.

### (b) Symmetry or range projection
Because each local expectation should lie in $[-1,1]$, clip or project mitigated values into the physical range.
For physically constrained outputs, also project reconstructed fields to admissible ranges when appropriate.

### (c) Zero-noise extrapolation (ZNE)
Fold entangling gates and extrapolate back to zero noise for the local-observable vector.
This is particularly helpful because the circuit is shallow and the observables are low-weight.

### (d) Pauli twirling / randomized compiling
Convert coherent entangler errors into more stochastic noise, which usually makes ZNE and averaging more reliable.

### (e) Sparse local measurement grouping
Because only 1-local observables are required, group commuting Pauli strings efficiently and avoid unnecessary measurement overhead.

### (f) Adaptive shot scheduling
Use low shots at the start, then increase only for batches or residual groups whose uncertainty blocks optimization.

### (g) Warm-start by noiseless simulation
First train in a noiseless or statevector regime, then fine-tune on hardware with mitigation and smaller learning rates.



## 12. Suggestions to improve the model beyond the original notebook

### Physics/modeling improvements
1. Add true **boundary-condition losses** for crucible wall, free surface, symmetry axis, and crystal interface.
2. Replace the steady formulation by a **time-dependent** one if interface evolution is important.
3. For concentration/temperature coupling, use a more explicit buoyancy term in momentum, e.g. Boussinesq forcing.
4. If this is meant to connect more directly to CZ growth, consider a **2D axisymmetric** formulation rather than plain Cartesian 2D.

### PINN/training improvements
1. Use **residual-based adaptive sampling** near the diffuse interface and near strong thermal gradients.
2. Nondimensionalize all PDEs to reduce gradient imbalance.
3. Use curriculum training: scalar flow first, then phase coupling, then full coupled loss.
4. Add Fourier features or multi-scale coordinate embeddings before the SI blocks.

### Quantum-side improvements
1. Keep depth $L$ small and increase qubit count only when the local feature bottleneck saturates.
2. Prefer **local-cost training** and local observables to avoid severe barren plateaus.
3. Use a **block-local ansatz** if hardware connectivity is limited.
4. For 16 qubits, measure only a compressed subset of qubits if post-network performance does not improve with all outputs.
5. Explore quantum feature reuse: one quantum feature vector can feed multiple PDE heads.



## 13. Recommended research framing for a paper

A strong paper-level claim is not "quantum is universally better than SI-LB-PINN." A stronger and more credible framing is:

> We construct a hybrid SI-LB-QPINN in which SI-style spatial enhancement and adaptive loss balancing are preserved, while a shallow local quantum latent map provides an additional nonlinear feature lift. The approach is especially attractive when using local observables and adaptive shot allocation, because the feature-estimation variance scales as $O(n_q/S)$ rather than directly with Hilbert-space dimension.

That framing is mathematically defensible, modest, and aligned with current hardware reality.


In [ ]:

# Example model construction (forward-pass prototype)
if QISKIT_AVAILABLE:
    qc, x_params, w_params = build_quantum_ansatz(
        n_qubits=N_QUBITS,
        n_layers=N_Q_LAYERS,
        feature_dim=LATENT_Q_DIM,
    )
    qlayer = QuantumFeatureExtractor(qc, x_params, w_params, N_QUBITS)
    model = HybridSILBQPINN(qlayer, N_QUBITS).to(DEVICE)

    x = torch.rand(4, 2, device=DEVICE, requires_grad=True)
    y = model(x)
    print("Circuit qubits:", qc.num_qubits)
    print("Quantum trainable angles:", len(w_params))
    print("Output shape:", y.shape)
else:
    print("Qiskit not installed here; architecture notebook generated successfully.")
